In [4]:
from jupyterscad import view

from configuration import load_config

In [5]:
conf = load_config()
conf

{'dist_u': 19.05, 'white_key_dims': {'length': 1.75, 'width': 1.25}, 'black_key_dims': {'length': 1.75, 'width': 1.0}, 'rows_height_diff_mm': 12.0, 'mount_plate_width': 1.25, 'mount_u': 14.0, 'keycap_u': 18.0, 'keycap_height_mm': 1.5, 'keycap_rounding_corner_mm': 2.0, 'output_dir': PosixPath('build'), 'white_black_keys_offset_mm': 6.0, 'base_height_mm': 8.0, 'connector_dims': {'base_diff_mm': 3.0, 'margin_mm': 0.16}, 'stand_r_mm': 5.0, 'stand_screw_r_mm': 0.5}

In [ ]:
from solid2 import cube

from connectors import generate_male_connector, generate_female_connector
from constants import WHITE_TO_BLACK_KEY_RATIO

from configuration import ConfigSchema
from common import generate_keys_row, generate_slope, generate_stand


def generate_kb_white_key_part(
    wk_total_width: float, octave_width: float, white_keys: float, conf: ConfigSchema
):
    white_plate_len = conf.white_key_dims.length * conf.dist_u
    wk_len_offset = (white_plate_len - conf.mount_u) / 2
    w_distances = [(wk_total_width - conf.mount_u) / 2] + [wk_total_width] * 7
    plate = (
        # upper wall, with mx mounting holes
        generate_keys_row(
            octave_width,
            white_plate_len,
            w_distances[:white_keys],
            wk_len_offset,
            conf,
        )
        +
        # front wall of the keyboard
        cube([octave_width, conf.mount_plate_width, conf.base_height_mm]).down(
            conf.base_height_mm
        )
        # slope added to the first wall - probably better to remove supports
        + generate_slope(
            octave_width - w_distances[0],
            wk_len_offset
            - conf.mount_plate_width
            - 1,  # minimal offset from mounting point to fit switch
            conf.base_height_mm,
        )
        .translateX(w_distances[0])
        .translateY(conf.mount_plate_width)
        .down(conf.base_height_mm)
        # connectors
        + generate_female_connector(w_distances[0], white_plate_len, conf)
        + generate_male_connector(w_distances[0], white_plate_len, conf).translateX(
            octave_width
        )
    )
    if white_keys > 1:
        plate += generate_stand(wk_total_width, white_plate_len, conf)
        plate += generate_stand(
            wk_total_width * (white_keys - 1), white_plate_len, conf
        )
    return plate.translate(
        [
            0,
            -white_plate_len,
            -conf.white_black_keys_offset_mm - conf.mount_plate_width,
        ]
    )


def generate_octave(white_keys: int, conf: ConfigSchema):
    assert white_keys <= 7

    wk_total_width = conf.white_key_dims.width * conf.dist_u
    octave_width = wk_total_width * white_keys

    bw_diff = conf.white_black_keys_offset_mm + conf.mount_plate_width
    b_distances = [
        wk_total_width - conf.mount_u / 2,
        wk_total_width,
        wk_total_width * 2,
        wk_total_width,
        wk_total_width,
    ]
    black_mount_plate = (
        generate_keys_row(
            octave_width,
            conf.dist_u,
            b_distances[: WHITE_TO_BLACK_KEY_RATIO[white_keys]],
            (conf.dist_u - conf.mount_u) / 2,
            conf,
        )
        + cube([octave_width, conf.mount_plate_width, bw_diff]).down(bw_diff)
        + cube([octave_width, conf.mount_plate_width, bw_diff + conf.base_height_mm])
        .down(bw_diff + conf.base_height_mm)
        .translateY(conf.dist_u - conf.mount_plate_width)
        + generate_female_connector(b_distances[0], conf.dist_u, conf)
        + generate_male_connector(b_distances[0], conf.dist_u, conf).translateX(
            octave_width
        )
    )
    return black_mount_plate + generate_kb_white_key_part(
        wk_total_width, octave_width, white_keys, conf
    )


small_af_octave = generate_octave(2, conf)
view(small_af_octave)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.7, position=(3.0, 5.0,…

In [ ]:
conf.output_dir.mkdir(parents=True, exist_ok=True)

small_af_octave.save_as_stl(
    conf.output_dir / "really_really_really_small_octave_v2.stl"
)

In [ ]:
# TODO: each octave is indivisible part
#       but it is possible to generate half of the octave
#       currently I have 35 switches, so the max I can get is 2.5 octaves.
#       30~ keys + 5 mods (octave up, octave down, maybe some play, record, etc)

